# 2.3 算子目录结构

本节介绍 CANN 算子库的标准目录组织，以及各层文件的作用。

---

## 学习目标

完成本节后，你将能够：
- 理解算子的标准目录结构
- 了解各层文件的作用和关联
- 理解 Tiling 的概念和常见实现内容
- 掌握 build.sh 的常用命令

---

## 1. 标准算子目录结构

以 FastGelu 为例，一个完整的算子工程包含以下目录和文件：

```
fast_gelu/
├── CMakeLists.txt              # 算子编译入口
├── op_host/                    # Host 侧：算子定义、推导、Tiling
│   ├── fast_gelu_def.cpp       # 算子定义（输入输出、数据类型、格式）
│   ├── fast_gelu_infershape.cpp # 形状推导
│   ├── arch35/                 # 架构相关（按芯片 SoC 版本分目录）
│   │   ├── fast_gelu_tiling_arch35.cpp
│   │   └── fast_gelu_tiling_arch35.h
│   └── config/
│       └── ascend950/
│           └── fast_gelu_binary.json
├── op_kernel/                  # Device 侧：Ascend C 核函数
│   ├── fast_gelu_apt.cpp       # Kernel 入口
│   └── arch35/
│       ├── fast_gelu_dag.h     # 计算 DAG 描述
│       └── fast_gelu_struct.h  # 数据结构定义
├── op_api/                     # API 层：对外接口
│   ├── fast_gelu.h / .cpp      # Level0 Executor 内部接口
│   └── aclnn_fast_gelu.h / .cpp # aclnn 两段式外部 API
├── op_graph/                   # 框架集成：IR 图注册
│   └── fast_gelu_proto.h
├── examples/                   # 示例程序
└── tests/                      # 测试
    ├── ut/                     # 单元测试
    └── st/                     # 系统测试
```

```mermaid
flowchart TB
    subgraph SRC["源码文件"]
        DEF["def.cpp<br/>算子定义"]
        INFER["infershape.cpp<br/>形状推导"]
        TILING["tiling.cpp<br/>数据切分"]
        APT["apt.cpp<br/>核函数"]
        API["op_api/"<br/>接口层]
    end
    subgraph BUILD["CMake 编译"]
        CMAKE["CMakeLists.txt<br/>add_modules_sources"]
    end
    subgraph OUTPUT["编译产物"]
        HOST["op_host 库<br/>host侧"]
        KERNEL["op_kernel 库<br/>device侧"]
        ACLNN["aclnn 接口<br/>对外API"]
    end
    DEF --> CMAKE
    INFER --> CMAKE
    TILING --> CMAKE
    APT --> CMAKE
    API --> CMAKE
    CMAKE --> HOST
    CMAKE --> KERNEL
    CMAKE --> ACLNN
```

---

## 2. 文件说明

### 2.1 CMakeLists.txt —— 编译入口

每个算子只有一个 CMakeLists.txt，通过 `add_modules_sources()` 函数声明支持的芯片和编译源：

```cmake
# 声明支持的芯片类型
set(SUPPORT_COMPUTE_UNIT "ascend950")
# 声明各芯片对应的 Tiling 目录
set(SUPPORT_TILING_DIR "arch35")
# 自动收集各子目录源文件并配置编译
add_modules_sources(HOSTNAME ${OPHOST_NAME}
    MODE PRIVATE
    COMPUTE_UNIT ${SUPPORT_COMPUTE_UNIT}
    TILING_DIR ${SUPPORT_TILING_DIR}
    OPTYPE fast_gelu)
```

- `SUPPORT_COMPUTE_UNIT` 声明支持的芯片，cmake 通过 `-DASCEND_COMPUTE_UNIT` 选择
- `SUPPORT_TILING_DIR` 声明对应的 tiling 代码目录
- 芯片区分**通过编译配置实现**，而非文件后缀

### 2.2 op_host/xxx_def.cpp —— 算子定义

定义算子的输入输出接口，包括参数数量、数据类型、格式，以及 Kernel 绑定：

```cpp
class XxxOp : public OpDef {
    explicit XxxOp(const char* name) : OpDef(name) {
        // 声明输入输出
        this->Input("x").ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND});
        this->Output("y").ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND});
        // 绑定 Kernel 文件（关联到 op_kernel/xxx_apt.cpp）
        OpAICoreConfig cfg;
        cfg.ExtendCfgInfo("opFile.value", "xxx_apt");
        this->AICore().AddConfig("ascend950", cfg);
    }
};
```

### 2.3 op_host/xxx_infershape.cpp —— 形状推导

推理算子输出的形状。Element-wise 算子可复用公共工具，非 element-wise 算子（Conv、Resize 等）需自行实现：

```cpp
// Element-wise：输入输出形状相同
IMPL_OP_INFERSHAPE(XxxOp).InferShape(Ops::Base::InferShape4Elewise);

// 自定义形状推导
IMPL_OP_INFERSHAPE(XxxOp).InferShape([](auto* ctx) {
    auto inputShape = ctx->GetInputShape(0);
    // ... 根据算子语义推导输出形状并设置
    return ge::GRAPH_SUCCESS;
});
```

### 2.4 op_host/tiling.cpp —— Tiling

Tiling（数据切分）是多核并行计算的关键环节。它将整体数据切分成多个小块，分配给多个 AI Core 并行处理，同时利用双缓冲机制在单核内实现搬运与计算的重叠。

**Tiling 的通用实现流程**：

```
1. 从 TilingContext 获取输入张量的形状、数据类型
2. 根据数据量和芯片资源计算分块参数
3. 将参数填入 TilingData 结构体，下发给 Kernel
```

**常见的 Tiling 参数**：

| 参数 | 说明 |
|------|------|
| `totalLength` | 待处理的数据总长度（元素数） |
| `blockDim` | 使用的 AI Core 数量，决定核间切分粒度 |
| `tileNum` | 每个核内的数据块个数，决定流水并行度 |
| `tileLength` | 每个数据块的元素数（= blockLength / tileNum / BUFFER_NUM） |
| `dataType` | 数据类型，用于 Kernel 侧模板实例化 |
| `workspaceSize` | 额外 Workspace 大小（部分算子需要临时缓冲区） |

**代码框架**：

```cpp
// xxx_tiling_data.h —— Tiling 数据结构
struct XxxTilingData {
    uint32_t totalLength;   // 数据总长度
    uint32_t blockDim;      // 核数
    uint32_t tileNum;       // 每核内的块数
    int32_t  dataType;      // 数据类型
};

// xxx_tiling.cpp —— Tiling 计算
ge::graphStatus TilingFunc(gert::TilingContext* context) {
    // 1. 获取输入信息
    auto xDesc = context->GetInputDesc(0);
    uint32_t totalLength = xDesc->GetShape().GetShapeSize();

    // 2. 计算分块参数
    XxxTilingData tiling;
    tiling.totalLength = totalLength;
    tiling.blockDim = CalculateBlockDim(totalLength);
    tiling.tileNum = 8;  // 每核 8 块，配合 BUFFER_NUM=2 得到 16 次 loop

    // 3. 下发给 Kernel
    context->SetTilingData(&tiling, sizeof(XxxTilingData));
    return ge::GRAPH_SUCCESS;
}
```

```mermaid
flowchart LR
    CTX["TilingContext<br/>获取输入信息"] --> S1["GetInputDesc<br/>获取shape"]
    S1 --> S2["计算 totalLength<br/>= shapeSize"]
    S2 --> S3["计算 blockDim<br/>= 核数"]
    S3 --> S4["计算 tileNum<br/>= 每核块数"]
    S4 --> S5["填充 TilingData<br/>totalLength + tileNum"]
    S5 --> S6["SetTilingData<br/>下发给 Kernel"]
    S6 --> K["Kernel 侧<br/>GET_TILING_DATA"]
```

Tiling 代码按芯片架构分目录（arch35 对应 ascend950），编译时由 `TILING_DIR` 指定。

### 2.5 op_kernel/xxx_apt.cpp —— Kernel 实现

Device 侧核函数，使用 Ascend C 实现计算逻辑：

```cpp
__global__ __aicore__ void xxx_kernel(GM_ADDR x, GM_ADDR y,
                                       GM_ADDR workspace, GM_ADDR tiling)
{
    GET_TILING_DATA_WITH_STRUCT(XxxTilingData, tilingData, tiling);
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    // ... 初始化 TPipe、TQue，执行 CopyIn/Compute/CopyOut
}
```

### 2.6 op_api/ —— 调用接口

两个层级：

| 文件 | 层级 | 用途 |
|------|------|------|
| `xxx.h / .cpp` | Level0 Executor | 内部 C++ 接口，由 op executor 框架调用 |
| `aclnn_xxx.h / .cpp` | aclnn 外部 API | 用户可调用的两段式 C 接口 |

---

## 3. build.sh 常用命令

以 ops-nn 仓为例：

```bash
# 整仓编译
bash build.sh                          # 全量编译
bash build.sh --soc=ascend950          # 指定芯片

# 单算子编译
bash build.sh --ops=fast_gelu          # 仅编译 host 库
bash build.sh --ops=fast_gelu --soc=ascend950

# 单算子出包
bash build.sh --pkg --soc=ascend950 --ops=fast_gelu

# 单元测试
bash build.sh -u                       # 全部 UT
bash build.sh -u --ops=fast_gelu       # 指定算子 UT
bash build.sh -u --ophost              # 仅 op_host UT

# 其他选项
bash build.sh --genop=activation/xxx   # 生成新算子骨架
bash build.sh --build-type=Debug       # Debug 模式编译
bash build.sh --run_example xxx eager   # 编译运行示例
```

```mermaid
flowchart TB
    subgraph 编译["编译阶段"]
        C1["bash build.sh<br/>--ops=fast_gelu"] --> C2["host 库编译"]
        C2 --> C3["kernel 库编译"]
    end
    subgraph 出包["出包阶段"]
        P1["bash build.sh --pkg<br/>--ops=fast_gelu"] --> P2["打包 .run 文件"]
    end
    subgraph 部署["部署阶段"]
        D1["./custom_opp.run<br/>--install-path=..."] --> D2["安装到指定目录"]
    end
    subgraph 测试["测试阶段"]
        T1["aclnn_test.cpp"] --> T2["两段式调用"]
        T2 --> T3["验证结果"]
    end
    编译 --> 出包 --> 部署 --> 测试
```

---

## 小结

本节介绍了算子的目录结构和关键文件：

| 文件 | 作用 |
|------|------|
| `CMakeLists.txt` | 声明芯片 + 自动收集源文件 |
| `op_host/def.cpp` | 输入输出定义 + Kernel 绑定 |
| `op_host/infershape.cpp` | 输出形状推导 |
| `op_host/tiling.cpp` | 数据切分参数计算 |
| `op_kernel/apt.cpp` | Ascend C 核函数入口 |
| `op_api/` | 内部接口 + aclnn 外部 API |

---

上一节：[2.2 算子执行流程](./02.02_operator_execution_flow.ipynb) | 下一节：[2.4 开发环境部署](./02.04_development_environment_setup.ipynb)